In [51]:
import pandas as pd
from camara_deputados.ingestion.data_loader import DataLoader
from camara_deputados.extraction.write import DataWrite
from camara_deputados.modeling.modeling import DataModeling
from camara_deputados.transformer.transformer import DataTransformer


In [52]:
# instâncias

silver = DataLoader('silver')
gold = DataLoader('gold')
salva= DataWrite()
transformer = DataTransformer()
modeling = DataModeling(transformer)


# Criando Dim na gold

In [95]:
dfs_silver = silver.carregar_parquets()


📂 Lendo dados da camada silver: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/notebooks/../data/silver

✅ dim_s_deputados: 513 linhas, 8 colunas
✅ dim_s_frente: 100 linhas, 4 colunas
✅ dim_s_partido: 15 linhas, 4 colunas
✅ dim_s_proposicao: 60 linhas, 7 colunas
✅ dim_s_tema: 21 linhas, 3 colunas
✅ dim_s_tiposAutor: 58 linhas, 5 colunas
✅ silver_autores: 57 linhas, 10 colunas
✅ silver_deputado: 513 linhas, 17 colunas
✅ silver_frenteDeputado: 131244 linhas, 5 colunas
✅ silver_proposica: 60 linhas, 22 colunas
✅ silver_proposicao: 60 linhas, 22 colunas
✅ silver_temasProposicao: 62 linhas, 6 colunas
✅ silver_tipos_autores: 58 linhas, 3 colunas
✅ silver_votacao_detalhamento: 271 linhas, 18 colunas
✅ silver_votacao_proposicao: 271 linhas, 9 colunas

🎯 Carregamento finalizado.


## Criando a dimensão dos deputados

In [54]:
df_s_deputados = dfs_silver['silver_deputado']



In [55]:
colunas_deputado = list(df_s_deputados.columns)
print(colunas_deputado)

['id_mandato', 'nom_NomeCivil', 'nom_Sexo', 'dat_DataNasc', 'dat_DataFalecimento', 'nom_UFNasc', 'nom_MunicipioNasci', 'nom_Escolaridade', 'nom_SiglaPartido', 'nom_UFRepresenta', 'id_legislatura', 'nom_Email', 'nom_NomeEleitoral', 'nom_Situacao', 'nom_CondEleitoral', 'id_deputado', 'data_extracao']


In [56]:
colunas_g_dim_dep = [#'id_mandato'
 'nom_NomeCivil'
, 'nom_Sexo'
, 'dat_DataNasc'
, 'dat_DataFalecimento'
, 'nom_UFNasc'
, 'nom_MunicipioNasci'
#, 'nom_Escolaridade'
#, 'nom_SiglaPartido'
#, 'nom_UFRepresenta'
#, 'id_Legislatura'
##, 'nom_NomeEleitoral'
#, 'nom_Situacao'
#, 'nom_CondEleitoral'
#, 'data_extracao'
, 'id_deputado'
]


In [57]:
dim_deputado = df_s_deputados[colunas_g_dim_dep].drop_duplicates(subset='id_deputado')

In [58]:
salva.save_parquet(dim_deputado, 'dim_deputado', 'gold')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/dim_deputado/dim_deputado.parquet


## Dim partidos

In [59]:
df_g_partido = modeling.criar_dim(
    df_s_deputados,
    colunas=['nom_SiglaPartido'],
    gerar_id=True,
    colunas_id=['nom_SiglaPartido'],
    nome_id='id_partido'
)


In [60]:
df_g_partido.head()

,nom_SiglaPartido,id_partido
0,MDB,6c152faa4bbca9a007fc7c608b6583a5
2,PL,9b7d173b068dc4d5517bfae92d676437
3,PSDB,952766f4c7293702128fa41f6466b9dc
4,NOVO,6ba66cf3ab9fce438581a0e4c84225d9
5,PP,4baee84af7a92177c9064ddc2fc010ce


In [61]:
salva.save_parquet(
    df=df_g_partido,
    dataset='dim_partido',
    layer='gold'
)

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/dim_partido/dim_partido.parquet


## Criando a dim mandato 

Ao levantar o detalhamento dos deputados, observado que as linhas representam mandatos dos deputados

In [62]:
coluna_mandato = ['id_mandato'
, 'nom_SiglaPartido'
, 'nom_UFRepresenta'
, 'id_legislatura'
, 'id_deputado'
]

df_dim_mandato = df_s_deputados[coluna_mandato].drop_duplicates(subset='id_mandato')

In [63]:
print(df_dim_mandato.head())
print(df_dim_mandato.columns)
df_dim_mandato.info()

   id_mandato nom_SiglaPartido nom_UFRepresenta  id_legislatura id_deputado
0      204379              MDB               AP              57      204379
1      220714              MDB               AM              57      220714
2      221328               PL               SP              57      221328
3      204560             PSDB               BA              57      204560
4      204528             NOVO               SP              57      204528
Index(['id_mandato', 'nom_SiglaPartido', 'nom_UFRepresenta', 'id_legislatura',
       'id_deputado'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 513 entries, 0 to 512
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id_mandato        513 non-null    Int64 
 1   nom_SiglaPartido  513 non-null    string
 2   nom_UFRepresenta  513 non-null    string
 3   id_legislatura    513 non-null    Int64 
 4   id_deputado       513 non-null    string
dtypes: 

In [64]:
#inclui o id do partido

df_mandato_final = df_dim_mandato.merge(
    df_g_partido,
    on="nom_SiglaPartido",
    how="left"
)

In [65]:
dim_mandato = modeling.criar_dim(
    df=df_mandato_final,
    colunas=[
        "id_mandato",
        "id_deputado",
        "id_partido",
        "id_legislatura",
        "nom_UFRepresenta"
    ],
    chave_duplicidade=["id_mandato"]
)

In [66]:
dim_mandato.head()

,id_mandato,id_deputado,id_partido,id_legislatura,nom_UFRepresenta
0,204379,204379,6c152faa4bbca9a007fc7c608b6583a5,57,AP
1,220714,220714,6c152faa4bbca9a007fc7c608b6583a5,57,AM
2,221328,221328,9b7d173b068dc4d5517bfae92d676437,57,SP
3,204560,204560,952766f4c7293702128fa41f6466b9dc,57,BA
4,204528,204528,6ba66cf3ab9fce438581a0e4c84225d9,57,SP


In [67]:
salva.save_parquet(
    df=dim_mandato,
    dataset='dim_mandato',
    layer='gold'
)

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/dim_mandato/dim_mandato.parquet


## DIM Proposição


In [68]:
df_silver_proposicao = dfs_silver['silver_proposica']

In [69]:
dim_proposicao = modeling.criar_dim(
    df=df_silver_proposicao,
    colunas=[
        "id_proposicao",
        "cod_Tipo",
        "nom_TipoProposicao",
        "num_NumeroProp",
        "num_ano",
        "nom_Ementa",
        "nom_regime",
        "dat_Apresentacao"
    ],
    chave_duplicidade=["id_proposicao"]
)

In [70]:
salva.save_parquet(
    df=dim_proposicao,
    dataset='dim_proposicao',
    layer='gold'
)

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/dim_proposicao/dim_proposicao.parquet


## Criando uma dim_temas

In [71]:
df_temas = transformer.split_and_explode(
    df_silver_proposicao,
    coluna="nom_Keywords",
    nova_coluna="tema"
)

In [72]:
df_temas.head()

,id_proposicao,nom_TipoProposicao,cod_Tipo,num_NumeroProp,num_ano,nom_Ementa,dat_Apresentacao,uri_Autor,nom_Keywords,nom_SiglaOrgao,...,cod_TipoTramitacao,cod_TipoSiituacao,cod_Situacao,nom_LinkProposicao,nom_TipoAutor,id_Autor,nom_TipoRelator,id_Relator,data_extracao,tema
0,369205,PL,139,4089,2023,Torna obrigatória a homologação em cartório de...,2007-09-26 12:37:00,https://dadosabertos.camara.leg.br/api/v2/prop...,"obrigatoriedade, homologação, cartório, reconh...",MESA,...,500,Aguardando Despacho do Presidente da Câmara do...,1201,https://www.camara.leg.br/proposicoesWeb/prop_...,proposicoes,369205,deputados,74856,2026-04-22 06:27:07.153182,obrigatoriedade
0,369205,PL,139,4089,2023,Torna obrigatória a homologação em cartório de...,2007-09-26 12:37:00,https://dadosabertos.camara.leg.br/api/v2/prop...,"obrigatoriedade, homologação, cartório, reconh...",MESA,...,500,Aguardando Despacho do Presidente da Câmara do...,1201,https://www.camara.leg.br/proposicoesWeb/prop_...,proposicoes,369205,deputados,74856,2026-04-22 06:27:07.153182,homologação
0,369205,PL,139,4089,2023,Torna obrigatória a homologação em cartório de...,2007-09-26 12:37:00,https://dadosabertos.camara.leg.br/api/v2/prop...,"obrigatoriedade, homologação, cartório, reconh...",MESA,...,500,Aguardando Despacho do Presidente da Câmara do...,1201,https://www.camara.leg.br/proposicoesWeb/prop_...,proposicoes,369205,deputados,74856,2026-04-22 06:27:07.153182,cartório
0,369205,PL,139,4089,2023,Torna obrigatória a homologação em cartório de...,2007-09-26 12:37:00,https://dadosabertos.camara.leg.br/api/v2/prop...,"obrigatoriedade, homologação, cartório, reconh...",MESA,...,500,Aguardando Despacho do Presidente da Câmara do...,1201,https://www.camara.leg.br/proposicoesWeb/prop_...,proposicoes,369205,deputados,74856,2026-04-22 06:27:07.153182,reconhecimento de firma
0,369205,PL,139,4089,2023,Torna obrigatória a homologação em cartório de...,2007-09-26 12:37:00,https://dadosabertos.camara.leg.br/api/v2/prop...,"obrigatoriedade, homologação, cartório, reconh...",MESA,...,500,Aguardando Despacho do Presidente da Câmara do...,1201,https://www.camara.leg.br/proposicoesWeb/prop_...,proposicoes,369205,deputados,74856,2026-04-22 06:27:07.153182,crédito consignado


In [73]:
dim_tema = modeling.criar_dim(
    df=df_temas,
    colunas=["tema"],
    gerar_id=True,
    colunas_id=["tema"],
    nome_id="id_tema"
)

In [74]:
dim_tema.head()

,tema,id_tema
0,obrigatoriedade,d6fd6cf76583ac2993421a47a941dd5f
0,homologação,11059e79a2527dbe57903cc9aea9bfbe
0,cartório,96b754baa134f491d8148c44d536fe67
0,reconhecimento de firma,ceb36a4e42124722da11823c1bbf493a
0,crédito consignado,8d329007599d6ca8ccebe0f5bcd0c3b6


# Bridge proposição tema

In [75]:
bridge_proposicao_tema = (
    df_temas
    .merge(dim_tema, on="tema", how="left")
    [["id_proposicao", "id_tema"]]
    .drop_duplicates()
)

salvando na gold

In [76]:
salva.save_parquet(dim_proposicao, "dim_proposicao", "gold")
salva.save_parquet(dim_tema, "dim_tema", "gold")
salva.save_parquet(bridge_proposicao_tema, "bridge_proposicao_tema", "gold")

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/dim_proposicao/dim_proposicao.parquet
💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/dim_tema/dim_tema.parquet
💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/bridge_proposicao_tema/bridge_proposicao_tema.parquet


## Dim_Autor

In [77]:
df_silver_autores = dfs_silver['silver_autores']

In [78]:
df_silver_autores.columns

Index(['uri_Autor', 'nom_Autor', 'cod_TipoAutor', 'nom_TipoAutor',
       'num_OrdemAssinatura', 'ind_Proponente', 'id_proposicao',
       'dat_Extracao', 'id_Autor', 'data_extracao'],
      dtype='str')

In [79]:
df_silver_autores.head()

,uri_Autor,nom_Autor,cod_TipoAutor,nom_TipoAutor,num_OrdemAssinatura,ind_Proponente,id_proposicao,dat_Extracao,id_Autor,data_extracao
0,https://dadosabertos.camara.leg.br/api/v2/depu...,Edgar Moury,10000,deputados,1,1,369205,2026-04-19 18:19:55.573900,141416,2026-05-01 10:19:27.869267
1,https://dadosabertos.camara.leg.br/api/v2/depu...,Giovani Cherini,10000,deputados,1,1,618609,2026-04-19 18:19:55.573900,160673,2026-05-01 10:19:27.869267
2,https://dadosabertos.camara.leg.br/api/v2/depu...,Evandro Rogerio Roman,10000,deputados,1,1,1197773,2026-04-19 18:19:55.573900,178930,2026-05-01 10:19:27.869267
3,https://dadosabertos.camara.leg.br/api/v2/depu...,Efraim Filho,10000,deputados,1,1,1198010,2026-04-19 18:19:55.573900,141422,2026-05-01 10:19:27.869267
4,https://dadosabertos.camara.leg.br/api/v2/depu...,Mara Gabrilli,10000,deputados,1,1,2074843,2026-04-19 18:19:55.573900,160565,2026-05-01 10:19:27.869267


In [80]:
df_silver_autores[['cod_TipoAutor','nom_TipoAutor','nom_Autor']].drop_duplicates()

,cod_TipoAutor,nom_TipoAutor,nom_Autor
0,10000,deputados,Edgar Moury
1,10000,deputados,Giovani Cherini
2,10000,deputados,Evandro Rogerio Roman
3,10000,deputados,Efraim Filho
4,10000,deputados,Mara Gabrilli
5,10000,deputados,André Figueiredo
6,30000,orgaos,Poder Executivo
12,40000,orgaos,Senado Federal - Roberto Muniz
14,10000,deputados,José Eduardo Cardozo
16,10000,deputados,Jaime Martins


In [81]:
df_silver_autores.info()


<class 'pandas.DataFrame'>
RangeIndex: 57 entries, 0 to 56
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   uri_Autor            57 non-null     string        
 1   nom_Autor            57 non-null     string        
 2   cod_TipoAutor        57 non-null     Int64         
 3   nom_TipoAutor        57 non-null     str           
 4   num_OrdemAssinatura  57 non-null     Int64         
 5   ind_Proponente       57 non-null     string        
 6   id_proposicao        57 non-null     Int64         
 7   dat_Extracao         57 non-null     datetime64[us]
 8   id_Autor             57 non-null     Int64         
 9   data_extracao        57 non-null     datetime64[us]
dtypes: Int64(4), datetime64[us](2), str(1), string(3)
memory usage: 9.1 KB


In [82]:
dim_autor = modeling.criar_dim(
    df=df_silver_autores,
    colunas=[
        "id_Autor",
        'nom_Autor',
        "cod_TipoAutor"
    ],
    chave_duplicidade=["id_Autor"]
)

In [83]:
dim_autor["id_Autor"] = dim_autor["id_Autor"].astype("Int64")
dim_deputado["id_deputado"] = dim_deputado["id_deputado"].astype("Int64")

In [84]:
df_validacao = dim_autor.merge(
    dim_deputado[["id_deputado"]],
    left_on="id_Autor",
    right_on="id_deputado",
    how="left",
    indicator=True
)


In [85]:
df_validacao.head(100)

,id_Autor,nom_Autor,cod_TipoAutor,id_deputado,_merge
0,141416,Edgar Moury,10000,<NA>,left_only
1,160673,Giovani Cherini,10000,160673,both
2,178930,Evandro Rogerio Roman,10000,<NA>,left_only
3,141422,Efraim Filho,10000,<NA>,left_only
4,160565,Mara Gabrilli,10000,<NA>,left_only
5,133439,André Figueiredo,10000,133439,both
6,253,Poder Executivo,30000,<NA>,left_only
7,78,Senado Federal - Roberto Muniz,40000,<NA>,left_only
8,74274,José Eduardo Cardozo,10000,<NA>,left_only
9,74665,Jaime Martins,10000,<NA>,left_only


## Dim Tipo Autor

In [86]:
df_dim_tipoAutor = dfs_silver['silver_tipos_autores']

In [87]:
df_dim_tipoAutor.columns

Index(['id_Autor', 'nom_NomeAutor', 'data_extracao'], dtype='str')

In [88]:
dim_tipoAutor = modeling.criar_dim(
    df_dim_tipoAutor,
    ['id_Autor', 'nom_NomeAutor'],
     chave_duplicidade=["id_Autor"]
)

## bridge proposicao autor

In [89]:
bridge_proposicao_autor = (
    df_silver_autores[["id_proposicao", "id_Autor"]]
    .drop_duplicates()
)

In [90]:
salva.save_parquet(dim_autor, "dim_autor", "gold")
salva.save_parquet(bridge_proposicao_autor, "bridge_proposicao_autor", "gold")

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/dim_autor/dim_autor.parquet
💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold/bridge_proposicao_autor/bridge_proposicao_autor.parquet


## Votações

In [96]:
df_votacoes_detalhamento = dfs_silver['silver_votacao_detalhamento']

In [99]:
df_votacoes_proposicao= dfs_silver['silver_votacao_proposicao']

In [100]:
df_votacoes_proposicao.info()

<class 'pandas.DataFrame'>
RangeIndex: 271 entries, 0 to 270
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id_votacao        0 non-null      Int64         
 1   dat_DataVotacao   271 non-null    datetime64[us]
 2   dat_DataRegistro  271 non-null    datetime64[us]
 3   uri_Orgao         271 non-null    string        
 4   nom_Descricao     271 non-null    string        
 5   ind_Aprovado      264 non-null    string        
 6   id_proposicao     271 non-null    Int64         
 7   id_Orgao          271 non-null    Int64         
 8   data_extracao     271 non-null    datetime64[us]
dtypes: Int64(3), datetime64[us](3), string(3)
memory usage: 54.5 KB


In [97]:
df_votacoes_detalhamento.info()

<class 'pandas.DataFrame'>
RangeIndex: 271 entries, 0 to 270
Data columns (total 18 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   id_Votacao                  0 non-null      Int64         
 1   dat_DataVotacao             271 non-null    datetime64[us]
 2   dat_DataHoraRegistro        271 non-null    datetime64[us]
 3   nom_SiglaOrgao              271 non-null    string        
 4   id_Orgao                    271 non-null    Int64         
 5   id_Evento                   206 non-null    Int64         
 6   des_DescricaoVotacao        271 non-null    string        
 7   ind_Aprovacao               264 non-null    string        
 8   des_UltimaAbertura          58 non-null     string        
 9   dat_UltimaAbertura          58 non-null     datetime64[us]
 10  des_Efeitos                 271 non-null    string        
 11  des_Objetos                 271 non-null    string        
 12  des_P

In [94]:
df_votacoes_proposicao.info()

<class 'pandas.DataFrame'>
RangeIndex: 271 entries, 0 to 270
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id_votacao        0 non-null      Int64         
 1   dat_DataVotacao   271 non-null    datetime64[us]
 2   dat_DataRegistro  271 non-null    datetime64[us]
 3   uri_Orgao         271 non-null    string        
 4   nom_Descricao     271 non-null    string        
 5   ind_Aprovado      264 non-null    string        
 6   id_proposicao     271 non-null    Int64         
 7   id_Orgao          271 non-null    Int64         
 8   data_extracao     271 non-null    datetime64[us]
dtypes: Int64(3), datetime64[us](3), string(3)
memory usage: 54.5 KB


In [101]:
df_votacoes_detalhamento.head()

,id_Votacao,dat_DataVotacao,dat_DataHoraRegistro,nom_SiglaOrgao,id_Orgao,id_Evento,des_DescricaoVotacao,ind_Aprovacao,des_UltimaAbertura,dat_UltimaAbertura,des_Efeitos,des_Objetos,des_Proposicoes,dat_UltimaApresentacao,des_UltimaApresentacaoDesc,des_UriProposicao,id_Proposicao,data_extracao
0,<NA>,2023-08-09,2023-08-09 21:59:25,PLEN,180,69153,Aprovada a Redação Final assinada pela Relator...,1.0,Votação da Redação Final.,2023-08-09 21:59:04,[],"[{'ano': 0, 'codTipo': 889, 'dataApresentacao'...","[{'ano': 2023, 'codTipo': 139, 'dataApresentac...",2023-08-10 13:04:07,Parecer às Emendas de Plenário proferido pela ...,https://dadosabertos.camara.leg.br/api/v2/prop...,<NA>,2026-05-01 10:20:23.152713
1,<NA>,2023-08-09,2023-08-09 21:58:06,PLEN,180,69153,Aprovada a Subemenda Substitutiva Global ao Pr...,1.0,Votação em turno único.,2023-08-09 21:46:57,[],"[{'ano': 0, 'codTipo': 889, 'dataApresentacao'...","[{'ano': 2023, 'codTipo': 139, 'dataApresentac...",2023-08-10 13:04:07,Parecer às Emendas de Plenário proferido pela ...,https://dadosabertos.camara.leg.br/api/v2/prop...,<NA>,2026-05-01 10:20:23.152713
2,<NA>,2023-08-01,2023-08-01 20:17:21,PLEN,180,<NA>,Alteração do Regime de Tramitação desta propos...,1.0,<NA>,NaT,[],"[{'ano': 2017, 'codTipo': 304, 'dataApresentac...","[{'ano': 2023, 'codTipo': 139, 'dataApresentac...",2017-07-05 12:16:17,Apresentação do Requerimento de Apensação n. 6...,https://dadosabertos.camara.leg.br/api/v2/prop...,<NA>,2026-05-01 10:20:23.152713
3,<NA>,2023-08-01,2023-08-01 20:17:21,SECAP(SGM),100001,<NA>,Realizar o encaminhamento do PL-2131/2007 à CI...,1.0,<NA>,NaT,[],"[{'ano': 2017, 'codTipo': 304, 'dataApresentac...","[{'ano': 2023, 'codTipo': 139, 'dataApresentac...",NaT,<NA>,<NA>,<NA>,2026-05-01 10:20:23.152713
4,<NA>,2023-08-01,2023-08-01 20:17:17,SECAP(SGM),100001,<NA>,Realizar o encaminhamento do PL-2131/2007 à CC...,1.0,<NA>,NaT,[],"[{'ano': 2017, 'codTipo': 304, 'dataApresentac...","[{'ano': 2023, 'codTipo': 139, 'dataApresentac...",NaT,<NA>,<NA>,<NA>,2026-05-01 10:20:23.152713
